# Contourdata voorbereiden

Dit notebook zet ruwe contourdata om naar invoerbestanden voor het dashboard (`input/lden_contour.csv` en `input/lnight_contour.csv`).

Per geluidscontour (1 dB-band) berekenen we onder meer bevolking, woningen, dosis–effect, vastgoedstock en maatregelstromen.

## Overzicht bronbestanden en kolommen

### Vlaanderen (+ rest land) — `data/contour.xlsx`

Eén rij per **1 dB-contourband** (bijv. `45-46`). Na hernoemen (`hernoem_vlaanderen_kolommen`):

| Kolom | Beschrijving |
|-------|----------------|
| `geluidscontour` | Label van de band (tekst, bijv. `45-46`) |
| `db_ondergrens` | Ondergrens van de band in dB |
| `db_bovengrens` | Bovengrens van de band in dB |
| `inwoners` | Inwoners in de contour (buiten Brussel) |
| `aantal_woningen` | Woningen in de contour (buiten Brussel) |
| `aantal_bebouwbare_percelen_woongebied` | Bebouwbare percelen door woongebied-aanwijzing (laatste 5 jaar) |
| `aantal_niet_bebouwbare_percelen_woongebied_schrapping` | Niet-bebouwbare percelen door schrapping woongebied |
| `prijs_onbebouwde_bebouwbare_percelen` | Prijs onbebouwd bebouwbare percelen |
| `prijs_onbebouwde_onbebouwbare_percelen` | Prijs onbebouwd onbebouwbare percelen |
| `prijs_bewoonde_niet_geïsoleerde_woning` | Prijs bewoonde niet-geïsoleerde woning |
| `prijs_bewoonde_geïsoleerde_woning` | Prijs bewoonde geïsoleerde woning |

### Brussel — `data/Population of brussels … 2024.xlsx`

Eerst per statistische sector, daarna **gesommeerd per dB**. Na hernoemen (`hernoem_brussel_kolommen`):

| Kolom | Beschrijving |
|-------|----------------|
| `db` | Geluidsniveau van de contour (enkelvoudig dB-getal, geen band) |
| `inwoners` | Inwoners in de contour (`Population dans le contour`) |

### Landelijk totaal — `df_lden` / `df_lnight` (na samenvoegen)

| Kolom | Beschrijving |
|-------|----------------|
| *(alle Vlaanderen-kolommen)* | Behouden; prijzen en percelen komen alleen uit Vlaanderen |
| `inwoners_vlaanderen` | Inwoners buiten Brussel (oorspronkelijke waarde) |
| `inwoners_brussel` | Brusselse inwoners gekoppeld aan de band (`db` = `db_ondergrens`) |
| `inwoners` | `inwoners_vlaanderen` + `inwoners_brussel` |
| `gemiddeld_aantal_inwoners_per_huis` | Ratio buiten Brussel (voor schatting woningen Brussel) |
| `aantal_woningen_vlaanderen` | Woningen buiten Brussel |
| `aantal_woningen_brussel` | Geschatte woningen Brussel (`inwoners_brussel` / gemiddelde bezetting) |
| `aantal_woningen_totaal` | Som Vlaanderen + Brussel |
| `aantal_ernstig_gehinderden_vlaanderen_2026` | Ernstig gehinderden buiten Brussel |
| `aantal_ernstig_gehinderden_brussel_2026` | Ernstig gehinderden in Brussel |
| `aantal_ernstig_gehinderden_totaal_2026` | Som Vlaanderen + Brussel |
| `aantal_<stock>_vlaanderen_2026` / `_brussel_2026` / `_totaal_2026` | Vastgoedstock per regio; simulatie op `*_totaal_*` |


In [210]:
import pandas as pd

from config import BEGINJAAR


def maak_regio_kolommen(
    df: pd.DataFrame,
    basis: str,
    jaar: int,
    vlaanderen: pd.Series,
    brussel: pd.Series,
) -> None:
    """Schrijf {basis}_vlaanderen_{jaar}, {basis}_brussel_{jaar}, {basis}_totaal_{jaar}."""
    df[f"{basis}_vlaanderen_{jaar}"] = vlaanderen
    df[f"{basis}_brussel_{jaar}"] = brussel
    df[f"{basis}_totaal_{jaar}"] = vlaanderen + brussel


def hernoem_vlaanderen_kolommen(df: pd.DataFrame) -> pd.DataFrame:
    """Vertaal en verkort kolomnamen uit contour.xlsx naar duidelijk Nederlands."""
    mapping = {
        "db_contour": "geluidscontour",
        "lower": "db_ondergrens",
        "upper": "db_bovengrens",
        "huizen": "aantal_woningen",
    }
    for kolom in df.columns:
        if "bebouwbare percelen die werden" in kolom:
            mapping[kolom] = "aantal_bebouwbare_percelen_woongebied"
        elif "niet-bebouwbare" in kolom or "woongebied te schrappen" in kolom:
            mapping[kolom] = "aantal_niet_bebouwbare_percelen_woongebied_schrapping"
    return df.rename(columns={k: v for k, v in mapping.items() if k in df.columns})


def hernoem_brussel_kolommen(df: pd.DataFrame) -> pd.DataFrame:
    """Vertaal Franse kolomnamen uit de Brussel-populatiedata."""
    return df.rename(
        columns={
            "dB": "db",
            "Population dans le contour": "inwoners",
        }
    )


def samenvoegen_vlaanderen_brussel(
    df_vlaanderen: pd.DataFrame,
    df_brussel: pd.DataFrame,
) -> pd.DataFrame:
    """
    Voeg landelijke contourdata samen met Brusselse inwoners.

    Brussel heeft één dB-waarde per rij; Vlaanderen heeft banden [db_ondergrens, db_bovengrens].
    Koppeling: Brusselse inwoners op niveau db worden toegevoegd aan de band waar db_ondergrens == db
    (bijv. Brussel 45 → band 45–46).
    """
    df = df_vlaanderen.copy()
    inwoners_per_db = df_brussel.set_index("db")["inwoners"]

    df["inwoners_vlaanderen"] = df["inwoners"]
    df["inwoners_brussel"] = df["db_ondergrens"].map(inwoners_per_db).fillna(0)
    df["inwoners"] = df["inwoners_vlaanderen"] + df["inwoners_brussel"]

    gemiddeld = df["gemiddeld_aantal_inwoners_per_huis"]
    df["aantal_woningen_vlaanderen"] = df["aantal_woningen"]
    df["aantal_woningen_brussel"] = df["inwoners_brussel"] / gemiddeld
    df["aantal_woningen_totaal"] = (
        df["aantal_woningen_vlaanderen"] + df["aantal_woningen_brussel"]
    )
    df = df.drop(columns=["aantal_woningen"])

    return df


## 1. Landelijke contourgegevens laden

Bron: `data/contour.xlsx` (tabbladen `lden` en `lnight`).

Elke rij is één **1 dB-geluidscontour** met onder meer grenzen van de band, inwoners, woningen en prijzen. Kolomnamen worden meteen hernoemd naar duidelijk Nederlands zodat de rest van het notebook leesbaar blijft.


In [211]:
df_lden_vlaanderen = pd.read_excel("data/contour.xlsx", sheet_name="lden")
df_lnight_vlaanderen = pd.read_excel("data/contour.xlsx", sheet_name="lnight")

df_lden_vlaanderen = hernoem_vlaanderen_kolommen(df_lden_vlaanderen)
df_lnight_vlaanderen = hernoem_vlaanderen_kolommen(df_lnight_vlaanderen)

df_lden_vlaanderen.head()

,geluidscontour,db_ondergrens,db_bovengrens,inwoners,aantal_woningen,aantal_bebouwbare_percelen_woongebied,aantal_niet_bebouwbare_percelen_woongebied_schrapping,prijs_onbebouwde_bebouwbare_percelen,prijs_onbebouwde_onbebouwbare_percelen,prijs_bewoonde_niet_geïsoleerde_woning,prijs_bewoonde_geïsoleerde_woning
0,45-46,45,46,45987,17688,0.6,4.4,200000,100000,400000,500000
1,46-47,46,47,28960,11714,1,4,197500,97500,395000,495000
2,47-48,47,48,27769,10989,0.6,2,195000,95000,390000,490000
3,48-49,48,49,38500,12806,0.2,3,192500,92500,385000,485000
4,49-50,49,50,26406,9891,0,2.8,190000,90000,380000,480000


## 2. Ontbrekende waarden opschonen

In de bron staan soms tekstwaarden `geen data`. Die vervangen we door `0` zodat berekeningen niet falen.


In [212]:
df_lden_vlaanderen.replace("geen data", 0, inplace=True)
df_lnight_vlaanderen.replace("geen data", 0, inplace=True)


,geluidscontour,db_ondergrens,db_bovengrens,inwoners,aantal_woningen,aantal_bebouwbare_percelen_woongebied,aantal_niet_bebouwbare_percelen_woongebied_schrapping,prijs_onbebouwde_bebouwbare_percelen,prijs_onbebouwde_onbebouwbare_percelen,prijs_bewoonde_niet_geïsoleerde_woning,prijs_bewoonde_geïsoleerde_woning
0,40-41,40,41,35414,12046,0.2,2.4,200000,100000,400000,500000
1,41-42,41,42,31633,11679,0,2.4,197500,97500,395000,495000
2,42-43,42,43,32185,11252,0,3.4,195000,95000,390000,490000
3,43-44,43,44,25275,8657,0,2.8,192500,92500,385000,485000
4,44-45,44,45,29155,9613,0,2.8,190000,90000,380000,480000
5,45-46,45,46,33567,11061,0,3.8,187500,87500,375000,475000
6,46-47,46,47,23487,7957,0,1.6,185000,85000,370000,470000
7,47-48,47,48,15552,5362,0.2,2.8,182500,82500,365000,465000
8,48-49,48,49,10335,3538,0.2,1.2,180000,80000,360000,460000
9,49-50,49,50,8694,2969,0.2,0.6,177500,77500,355000,455000


## 3. Bevolking Brussels Hoofdstedelijk Gewest

Bron: `data/Population of brussels per statistical sector and db contour 2024.xlsx`.

De data staan per **statistische sector**; dezelfde dB-waarde komt dus meerdere keren voor. We sommeren `inwoners` per `db` zodat één rij overblijft per geluidsniveau — analoog aan de landelijke contourbanden.


In [213]:
df_lden_brussels = pd.read_excel(
    "data/Population of brussels per statistical sector and db contour 2024.xlsx",
    sheet_name="lden",
)
df_lnight_brussels = pd.read_excel(
    "data/Population of brussels per statistical sector and db contour 2024.xlsx",
    sheet_name="lnight",
)

df_lden_brussels = hernoem_brussel_kolommen(
    df_lden_brussels[["dB", "Population dans le contour"]]
)
df_lnight_brussels = hernoem_brussel_kolommen(
    df_lnight_brussels[["dB", "Population dans le contour"]]
)

In [214]:
df_lden_brussels = df_lden_brussels.groupby("db", as_index=False)["inwoners"].sum()
df_lnight_brussels = df_lnight_brussels.groupby("db", as_index=False)["inwoners"].sum()

df_lden_brussels.head()

,db,inwoners
0,45,105153.667771
1,46,127964.973017
2,47,94698.794036
3,48,111977.055565
4,49,103292.390429


## 4. Schatting aantal woningen in het Brussels Hoofdstedelijk Gewest

Voor Brussel ontbreken woningaantallen per sector. We schatten het aantal woningen per geluidscontour via het **gemiddeld aantal inwoners per woning** buiten Brussel (`inwoners` / `aantal_woningen` in `contour.xlsx`).

**Stap 1 — gemiddelde bezetting buiten Brussel** (per contourband):

$$
\bar{i}_{\text{buiten Brussel}} = \frac{I_{\text{buiten Brussel}}}{W_{\text{buiten Brussel}}}
$$

**Stap 2 — geschat aantal woningen per contour** (Brussel + rest van het land):

$$
\hat{W}_{\text{contour}} = \frac{I_{\text{Brussel}} + I_{\text{buiten Brussel}}}{\bar{i}_{\text{buiten Brussel}}}
$$

| Symbool | Kolom in dit notebook |
|--------|------------------------|
| $I_{\text{Brussel}}$ | `inwoners` in `df_*_brussels` |
| $I_{\text{buiten Brussel}}$ | `inwoners` in `df_lden_vlaanderen` / `df_lnight_vlaanderen` |
| $W_{\text{buiten Brussel}}$ | `aantal_woningen` in de Vlaanderen-dataframes |
| $\hat{W}_{\text{contour}}$ | `aantal_woningen_totaal` na `samenvoegen_vlaanderen_brussel` (§5) |


In [215]:
df_lden_vlaanderen["gemiddeld_aantal_inwoners_per_huis"] = (
    df_lden_vlaanderen["inwoners"] / df_lden_vlaanderen["aantal_woningen"]
)
df_lnight_vlaanderen["gemiddeld_aantal_inwoners_per_huis"] = (
    df_lnight_vlaanderen["inwoners"] / df_lnight_vlaanderen["aantal_woningen"]
)

## 5. Samenvoegen Vlaanderen en Brussel

Vlaanderen levert de volledige contourtabel (woningen, prijzen, percelen). Brussel levert enkel **inwoners per dB**.

- **Inwoners:** per band optellen (`inwoners_vlaanderen` + `inwoners_brussel`).
- **Woningen:** buiten Brussel blijven de gemeten waarden; voor Brussel schatten we woningen met de gemiddelde bezetting buiten Brussel (zie §4).
- **Koppeling dB:** `db` (Brussel) = `db_ondergrens` (Vlaanderen).

Het resultaat is `df_lden` / `df_lnight` voor alle verdere stappen.

In [216]:
df_lden = samenvoegen_vlaanderen_brussel(df_lden_vlaanderen, df_lden_brussels)
df_lnight = samenvoegen_vlaanderen_brussel(df_lnight_vlaanderen, df_lnight_brussels)

df_lden[
    [
        "geluidscontour",
        "db_ondergrens",
        "db_bovengrens",
        "inwoners_vlaanderen",
        "inwoners_brussel",
        "inwoners",
        "aantal_woningen_vlaanderen",
        "aantal_woningen_brussel",
        "aantal_woningen_totaal",
    ]
].head()

,geluidscontour,db_ondergrens,db_bovengrens,inwoners_vlaanderen,inwoners_brussel,inwoners,aantal_woningen_vlaanderen,aantal_woningen_brussel,aantal_woningen_totaal
0,45-46,45,46,45987,105153.667771,151140.667771,17688,40445.301401,58133.301401
1,46-47,46,47,28960,127964.973017,156924.973017,11714,51760.417608,63474.417608
2,47-48,47,48,27769,94698.794036,122467.794036,10989,37475.063836,48464.063836
3,48-49,48,49,38500,111977.055565,150477.055565,12806,37246.186326,50052.186326
4,49-50,49,50,26406,103292.390429,129698.390429,9891,38690.639769,48581.639769


## 5. Dosis–effect en ernstig gehinderden

Voor elke contour nemen we het **midden van de dB-band** (`db_midden`) en passen daarop de dosis–effectformule toe (andere coëfficiënten voor Lden en Lnight).

Het aantal ernstig gehinderden per regio is `inwoners × dosis_effect_relatie` (zelfde relatie per band). Het totaal is de som van Vlaanderen+land en Brussel.


### Benodigde outputkolommen (checklist)

- [x] Geluidscontouren en dB (`geluidscontour`, `db_ondergrens`, `db_bovengrens`)
- [x] Aantal inwoners per geluidscontour
- [x] Aantal woningen per geluidscontour (`aantal_woningen_vlaanderen`, `_brussel`, `_totaal`)
- [x] Gemiddeld aantal inwoners per woning
- [x] Dosis-effectrelatie (Lden vs Lnight)
- [x] Aantal ernstig gehinderden
- [x] Bebouwbare percelen door woongebied (`aantal_bebouwbare_percelen_woongebied`)
- [ ] Prijzen en overige vastgoedstocks (deels voorbeelddata hieronder)


In [217]:
def dosis_effect_relatie(db, teller_1, teller_2, teller_3, noemer):
    return (teller_1 + teller_2 * db + pow(db, 2) * teller_3) / noemer


def lden_dosis_effect_relatie(db):
    return dosis_effect_relatie(db, -50.9693, 1.0168, 0.0072, 100)


def lnight_dosis_effect_relatie(db):
    return dosis_effect_relatie(db, 16.7885, -0.9293, 0.0198, 100)

In [218]:
df_lden["db_midden"] = (df_lden["db_ondergrens"] + df_lden["db_bovengrens"]) / 2
df_lnight["db_midden"] = (df_lnight["db_ondergrens"] + df_lnight["db_bovengrens"]) / 2

df_lden["dosis_effect_relatie"] = df_lden["db_midden"].apply(lden_dosis_effect_relatie)
df_lnight["dosis_effect_relatie"] = df_lnight["db_midden"].apply(lnight_dosis_effect_relatie)

In [219]:
for df in (df_lden, df_lnight):
    maak_regio_kolommen(
        df,
        "aantal_ernstig_gehinderden",
        BEGINJAAR,
        df["inwoners_vlaanderen"] * df["dosis_effect_relatie"],
        df["inwoners_brussel"] * df["dosis_effect_relatie"],
    )

## 6. Vastgoedstock (voorbeelddata)

De onderstaande aantallen zijn **plaatsvervangers** op basis van `aantal_woningen_vlaanderen` / `aantal_woningen_brussel` (vaste percentages) tot echte stockdata beschikbaar zijn.

Per stock: `aantal_<stock>_vlaanderen_{jaar}`, `_brussel_{jaar}`, `_totaal_{jaar}`. De simulatie leest enkel `*_totaal_*`.


In [220]:
# Voorbeeldverdeling vastgoed per regio — te vervangen door echte data
PCT_GEÏSOLEERD = 0.20
PCT_NIET_GEÏSOLEERD = 0.80
PCT_ONBEBOUWDE_BEBOUWBAAR = 0.20

for df in (df_lden, df_lnight):
    won_v = df["aantal_woningen_vlaanderen"]
    won_b = df["aantal_woningen_brussel"]

    maak_regio_kolommen(
        df,
        "aantal_bewoonde_geïsoleerde_huizen",
        BEGINJAAR,
        won_v * PCT_GEÏSOLEERD,
        won_b * PCT_GEÏSOLEERD,
    )
    maak_regio_kolommen(
        df,
        "aantal_bewoonde_niet_geïsoleerde_huizen",
        BEGINJAAR,
        won_v * PCT_NIET_GEÏSOLEERD,
        won_b * PCT_NIET_GEÏSOLEERD,
    )
    maak_regio_kolommen(
        df,
        "aantal_onbebouwde_bebouwbare_percelen",
        BEGINJAAR,
        won_v * PCT_ONBEBOUWDE_BEBOUWBAAR,
        won_b * PCT_ONBEBOUWDE_BEBOUWBAAR,
    )
    maak_regio_kolommen(
        df,
        "aantal_onbebouwde_onbebouwbare_percelen",
        BEGINJAAR,
        df["aantal_niet_bebouwbare_percelen_woongebied_schrapping"],
        0.0,
    )
    maak_regio_kolommen(
        df,
        "aantal_perceel_eigendom_overheid",
        BEGINJAAR,
        0.0,
        0.0,
    )
    maak_regio_kolommen(
        df,
        "aantal_woning_eigendom_overheid",
        BEGINJAAR,
        0.0,
        0.0,
    )

## Grafieken maken

Grafieken per geluidscontour voor **Lden** en **Lnight**, met het Idea Consult Altair-thema (`idea_consult_altair_theme.py`).

Per indicator, drie grafieken:
1. **Totaal** — inwoners en ernstig gehinderden (twee lijnen)
2. **Gestapeld** — inwoners Vlaanderen + Brussel
3. **Gestapeld** — ernstig gehinderden Vlaanderen + Brussel

Voer eerst de cellen over samenvoegen, dosis–effect en `aantal_ernstig_gehinderden_*_2026` uit.

In [221]:
import altair as alt
import pandas as pd

import idea_consult_altair_theme  # noqa: F401 — registreert en activeert het Idea Consult-thema

from idea_consult_altair_theme import CATEGORY_COLORS

# --- As en hulp (x-as per indicator: eigen onder-/bovengrens) ---
grafiek_id_kolommen = [
    "geluidscontour",
    "db_ondergrens",
    "db_bovengrens",
    "db_midden",
]


def db_x_as(df: pd.DataFrame) -> alt.X:
    """X-as op db_midden, domein = min ondergrens t/m max bovengrens van deze contour."""
    db_min = int(df["db_ondergrens"].min())
    db_max = int(df["db_bovengrens"].max())
    ticks = list(range(db_min, db_max + 1))
    return alt.X(
        "db_midden:Q",
        title="Geluidsniveau (dB)",
        scale=alt.Scale(domain=[db_min, db_max], nice=False),
        axis=alt.Axis(values=ticks, format="d"),
    )

regio_kleuren = alt.Scale(
    domain=["Vlaanderen", "Brussel"],
    range=CATEGORY_COLORS[:2],
)

regio_tooltips = [
    alt.Tooltip("geluidscontour:N", title="Contour"),
    alt.Tooltip("db_ondergrens:Q", title="dB ondergrens", format="d"),
    alt.Tooltip("db_bovengrens:Q", title="dB bovengrens", format="d"),
    alt.Tooltip("db_midden:Q", title="dB midden", format=".1f"),
    alt.Tooltip("regio:N", title="Regio"),
    alt.Tooltip("aantal:Q", title="Aantal", format=",.0f"),
]


def gestapelde_regio_lijngrafiek(
    df: pd.DataFrame,
    kolommen: dict[str, str],
    titel: str,
    x_db: alt.X,
    y_titel: str = "Aantal personen",
) -> alt.Chart:
    data = (
        df[grafiek_id_kolommen + list(kolommen.keys())]
        .sort_values("db_midden")
        .melt(
            id_vars=grafiek_id_kolommen,
            value_vars=list(kolommen.keys()),
            var_name="regio_kolom",
            value_name="aantal",
        )
    )
    data["regio"] = data["regio_kolom"].map(kolommen)

    return (
        alt.Chart(data)
        .mark_line(point=True)
        .encode(
            x=x_db,
            y=alt.Y("aantal:Q", stack="zero", title=y_titel, axis=alt.Axis(format=",.0f")),
            color=alt.Color(
                "regio:N",
                title="Regio",
                scale=regio_kleuren,
                sort=["Vlaanderen", "Brussel"],
            ),
            order=alt.Order("regio:N", sort="ascending"),
            tooltip=regio_tooltips,
        )
        .properties(title=titel)
    )


totaal_tooltips = [
    alt.Tooltip("geluidscontour:N", title="Contour"),
    alt.Tooltip("db_ondergrens:Q", title="dB ondergrens", format="d"),
    alt.Tooltip("db_bovengrens:Q", title="dB bovengrens", format="d"),
    alt.Tooltip("db_midden:Q", title="dB midden", format=".1f"),
    alt.Tooltip("reeks:N", title="Reeks"),
    alt.Tooltip("aantal:Q", title="Aantal", format=",.0f"),
]


def maak_grafiek_set(df: pd.DataFrame, indicator: str) -> dict[str, alt.Chart]:
    """Bouw de drie contourgrafieken voor Lden of Lnight (eigen x-asbereik)."""
    x_db = db_x_as(df)
    grafiek_data = (
        df[grafiek_id_kolommen + ["inwoners", "aantal_ernstig_gehinderden_totaal_2026"]]
        .sort_values("db_midden")
        .melt(
            id_vars=grafiek_id_kolommen,
            value_vars=["inwoners", "aantal_ernstig_gehinderden_totaal_2026"],
            var_name="reeks",
            value_name="aantal",
        )
    )
    grafiek_data["reeks"] = grafiek_data["reeks"].map(
        {
            "inwoners": "Inwoners",
            "aantal_ernstig_gehinderden_totaal_2026": "Ernstig gehinderden",
        }
    )

    totaal = (
        alt.Chart(grafiek_data)
        .mark_line(point=True)
        .encode(
            x=x_db,
            y=alt.Y("aantal:Q", title="Aantal personen", axis=alt.Axis(format=",.0f")),
            color=alt.Color(
                "reeks:N",
                title=None,
                scale=alt.Scale(
                    domain=["Inwoners", "Ernstig gehinderden"],
                    range=CATEGORY_COLORS[:2],
                ),
            ),
            tooltip=totaal_tooltips,
        )
        .properties(
            title=f"Inwoners en ernstig gehinderden per geluidscontour ({indicator})"
        )
    )

    inwoners_gestapeld = gestapelde_regio_lijngrafiek(
        df,
        {"inwoners_vlaanderen": "Vlaanderen", "inwoners_brussel": "Brussel"},
        titel=f"Inwoners per geluidscontour ({indicator}) — Vlaanderen en Brussel",
        x_db=x_db,
    )

    gehinderden_gestapeld = gestapelde_regio_lijngrafiek(
        df,
        {
            "aantal_ernstig_gehinderden_vlaanderen_2026": "Vlaanderen",
            "aantal_ernstig_gehinderden_brussel_2026": "Brussel",
        },
        titel=f"Ernstig gehinderden per geluidscontour ({indicator}) — Vlaanderen en Brussel",
        x_db=x_db,
    )

    return {
        "totaal": totaal,
        "inwoners_gestapeld": inwoners_gestapeld,
        "gehinderden_gestapeld": gehinderden_gestapeld,
    }


grafieken_lden = maak_grafiek_set(df_lden, "Lden")
grafieken_lnight = maak_grafiek_set(df_lnight, "Lnight")

# Voorbeeld in notebook: Lden-totaal (x-as 45–75); Lnight heeft eigen bereik (bijv. 40–70)
grafieken_lden["totaal"]


alt.Chart(...)

## Grafieken exporteren naar PNG

Zes losse PNG’s in `output/grafieken/` (`lden_*.png` en `lnight_*.png`, elk drie grafieken). Geen gecombineerde PNG’s. Lden en Lnight hebben elk een eigen x-asbereik (eigen onder-/bovengrens). Vereiste: `pip install vl-convert-python` (staat in `requirements.txt`).

In [222]:
from pathlib import Path

GRAFIEKEN_MAP = Path("output/grafieken")
GRAFIEKEN_MAP.mkdir(parents=True, exist_ok=True)


def exporteer_grafiek_naar_png(
    chart: alt.Chart,
    bestandsnaam: str,
    scale_factor: float = 2,
) -> Path:
    """Sla een Altair-grafiek op als PNG (via vl-convert)."""
    pad = GRAFIEKEN_MAP / bestandsnaam
    chart.save(str(pad), scale_factor=scale_factor)
    print(f"Opgeslagen: {pad.resolve()}")
    return pad


grafieken_te_exporteren: dict[str, alt.Chart] = {}
for prefix, grafiek_set in [
    ("lden", grafieken_lden),
    ("lnight", grafieken_lnight),
]:
    for sleutel, grafiek in grafiek_set.items():
        grafieken_te_exporteren[f"{prefix}_{sleutel}.png"] = grafiek

for naam, grafiek in grafieken_te_exporteren.items():
    exporteer_grafiek_naar_png(grafiek, naam)

Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Financiering van de maatregelen ‘RO en beheer’ als 2e pijler van de Ba)\dashboard-balanced-approach\output\grafieken\lden_totaal.png
Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Financiering van de maatregelen ‘RO en beheer’ als 2e pijler van de Ba)\dashboard-balanced-approach\output\grafieken\lden_inwoners_gestapeld.png
Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Financiering van de maatregelen ‘RO en beheer’ als 2e pijler van de Ba)\dashboard-balanced-approach\output\grafieken\lden_gehinderden_gestapeld.png
Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Financiering van de maatregelen ‘RO en beheer’ als 2e pijler van de Ba)\dashboard-balanced-approach\output\grafieken\lnight_totaal.png
Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Fina

## 7. Exporteren naar het dashboard

De dataframes worden weggeschreven naar `input/`. Het dashboard koppelt zones via `db_midden` en leest stockkolommen `aantal_<stock>_totaal_{BEGINJAAR}` (regionale opsplitsing blijft op het contour).


db_contour,lower,upper,inwoners,huizen,aantal bebouwbare percelen die werden gecreëerd door woongebieden aan te duiden,aantal niet-bebouwbarepercelen die worden gecreëerd door woongebied te schrappen,prijs_onbebouwde_bebouwbare_percelen,prijs_onbebouwde_onbebouwbare_percelen,prijs_bewoonde_niet_geïsoleerde_woning,prijs_bewoonde_geïsoleerde_woning,midden,dosis_effect_relatie,gemiddeld_aantal_inwoners_per_huis,aantal_ernstig_gehinderden_2026,aantal_bewoonde_geïsoleerde_huizen_2026,aantal_bewoonde_niet_geïsoleerde_huizen_2026,aantal_onbebouwde_bebouwbare_percelen_2026,aantal_perceel_eigendom_overheid_2026,aantal_woning_eigendom_overheid

In [223]:
df_lden.columns

Index(['geluidscontour', 'db_ondergrens', 'db_bovengrens', 'inwoners',
       'aantal_bebouwbare_percelen_woongebied',
       'aantal_niet_bebouwbare_percelen_woongebied_schrapping',
       'prijs_onbebouwde_bebouwbare_percelen',
       'prijs_onbebouwde_onbebouwbare_percelen',
       'prijs_bewoonde_niet_geïsoleerde_woning',
       'prijs_bewoonde_geïsoleerde_woning',
       'gemiddeld_aantal_inwoners_per_huis', 'inwoners_vlaanderen',
       'inwoners_brussel', 'aantal_woningen_vlaanderen',
       'aantal_woningen_brussel', 'aantal_woningen_totaal', 'db_midden',
       'dosis_effect_relatie', 'aantal_ernstig_gehinderden_vlaanderen_2026',
       'aantal_ernstig_gehinderden_brussel_2026',
       'aantal_ernstig_gehinderden_totaal_2026',
       'aantal_bewoonde_geïsoleerde_huizen_vlaanderen_2026',
       'aantal_bewoonde_geïsoleerde_huizen_brussel_2026',
       'aantal_bewoonde_geïsoleerde_huizen_totaal_2026',
       'aantal_bewoonde_niet_geïsoleerde_huizen_vlaanderen_2026',
       'aant

In [224]:
df_lden.to_csv("input/lden_contour.csv")
df_lnight.to_csv("input/lnight_contour.csv")

## 8. Intensiteit maatregelstromen (flows)

Per maatregel berekenen we in- en uitstroomintensiteit per contour. Onderstaande formules zijn een **eerste schets** (woongebiedverbod / aankoopbeleid); de noemer gebruikt voorlopig de indicator voor niet-bebouwbare percelen bij woongebiedschrapping.


In [225]:
# Woongebiedverbod — intensiteit = aandeel bebouwbare percelen / referentiestock
""" _bebouwbaar = df_lden["aantal_bebouwbare_percelen_woongebied"]
_referentie = df_lden["aantal_onbebouwde_onbebouwbare_percelen_totaal_2026"]

df_lden["woongebied_verbod_inflow_zonder_maatregel"] = _bebouwbaar / _referentie
df_lden["woongebied_verbod_outflow_zonder_maatregel"] = _bebouwbaar / _referentie
df_lden["woongebied_verbod_inflow_met_maatregel"] = 0
df_lden["woongebied_verbod_outflow_met_maatregel"] = 0

# Aankoopbeleid (voorbeeld — zelfde structuur, andere maatregel)
df_lden["aankoopbeleid_inflow_zonder_maatregel"] = 0
df_lden["aankoopbeleid_outflow_zonder_maatregel"] = 0
df_lden["aankoopbeleid_inflow_met_maatregel"] = _bebouwbaar / _referentie
df_lden["aankoopbeleid_outflow_met_maatregel"] = _bebouwbaar / _referentie """

' _bebouwbaar = df_lden["aantal_bebouwbare_percelen_woongebied"]\n_referentie = df_lden["aantal_onbebouwde_onbebouwbare_percelen_totaal_2026"]\n\ndf_lden["woongebied_verbod_inflow_zonder_maatregel"] = _bebouwbaar / _referentie\ndf_lden["woongebied_verbod_outflow_zonder_maatregel"] = _bebouwbaar / _referentie\ndf_lden["woongebied_verbod_inflow_met_maatregel"] = 0\ndf_lden["woongebied_verbod_outflow_met_maatregel"] = 0\n\n# Aankoopbeleid (voorbeeld — zelfde structuur, andere maatregel)\ndf_lden["aankoopbeleid_inflow_zonder_maatregel"] = 0\ndf_lden["aankoopbeleid_outflow_zonder_maatregel"] = 0\ndf_lden["aankoopbeleid_inflow_met_maatregel"] = _bebouwbaar / _referentie\ndf_lden["aankoopbeleid_outflow_met_maatregel"] = _bebouwbaar / _referentie '